#Data Preprocessing pipeline

##Imports

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

if on Google Colab

In [ ]:
# Google Colab Drive mounting removed for GitHub portability.


##Project Paths

###Google Colab

In [ ]:
project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
project_path = project_path.resolve()

print("Project path:", project_path)


For the local GitHub project:

In [ ]:
project_path = Path.cwd()

In [ ]:
dataset_dir = project_path / "datasets"
processed_dir = dataset_dir / "processed"

train_raw_path = dataset_dir / "training_data.csv"
test_raw_path = dataset_dir / "testing_data.csv"

train_cleaned_path = processed_dir / "train_cleaned.csv"
test_cleaned_path = processed_dir / "test_cleaned.csv"
report_path = processed_dir / "preprocessing_report.json"

processed_dir.mkdir(parents=True, exist_ok=True)

##Download NLTK Resources

In [ ]:
required_nltk_packages = [
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "stopwords",
    "punkt",
    "punkt_tab",
]

for package in required_nltk_packages:
    nltk.download(package)

##Helper Functions

In [ ]:
def get_wordnet_pos_from_tag(tag):
    first_letter = tag[0]

    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV,
    }

    return tag_dict.get(first_letter, wordnet.NOUN)

def clean_headline(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_text(text, lemmatizer, stop_words):
    clean_text = clean_headline(text)
    tokens = word_tokenize(clean_text)
    tagged_tokens = nltk.pos_tag(tokens)

    lemmatized_tokens = [
        lemmatizer.lemmatize(word, get_wordnet_pos_from_tag(pos))
        for word, pos in tagged_tokens
    ]

    final_tokens = [
        word for word in lemmatized_tokens
        if word not in stop_words
    ]

    return {
        "clean_text": clean_text,
        "tokens_json": json.dumps(final_tokens),
        "processed_text": " ".join(final_tokens),
    }

def preprocess_dataframe(df, text_column, lemmatizer, stop_words):
    processed_rows = df[text_column].apply(
        lambda text: preprocess_text(text, lemmatizer, stop_words)
    )

    processed_df = pd.DataFrame(processed_rows.tolist())

    return pd.concat(
        [df.reset_index(drop=True), processed_df],
        axis=1,
    )

##Load Raw Data

In [ ]:
if not train_raw_path.exists():
    raise FileNotFoundError(f"Missing training file: {train_raw_path}")

if not test_raw_path.exists():
    raise FileNotFoundError(f"Missing testing file: {test_raw_path}")

train_df = pd.read_csv(
    train_raw_path,
    sep="	",
    header=None,
    names=["label", "headline"],
    encoding="utf-8-sig",
)

test_df = pd.read_csv(
    test_raw_path,
    sep="	",
    header=None,
    names=["original_label", "headline"],
    encoding="utf-8-sig",
)


##Validate Raw Data

In [ ]:
required_train_columns = {"label", "headline"}
missing_train_columns = required_train_columns - set(train_df.columns)

if missing_train_columns:
    raise ValueError(f"Missing training columns: {missing_train_columns}")

required_test_columns = {"original_label", "headline"}
missing_test_columns = required_test_columns - set(test_df.columns)

if missing_test_columns:
    raise ValueError(f"Missing testing columns: {missing_test_columns}")

train_df["headline"] = train_df["headline"].astype(str).str.strip()
test_df["headline"] = test_df["headline"].astype(str).str.strip()

train_df = train_df.dropna(subset=["label", "headline"])
test_df = test_df.dropna(subset=["headline"])

train_df = train_df[train_df["headline"] != ""]
test_df = test_df[test_df["headline"] != ""]

train_df["label"] = train_df["label"].astype(int)

valid_labels = {0, 1}
bad_labels = set(train_df["label"].unique()) - valid_labels

if bad_labels:
    raise ValueError(f"Unexpected labels found: {bad_labels}")

##Inspect data

In [ ]:
print("Raw train shape:", train_df.shape)
print("Raw test shape:", test_df.shape)

print("\nTrain label counts:")
print(train_df["label"].value_counts())

print("\nMissing values in train:")
print(train_df.isna().sum())

print("\nMissing values in test:")
print(test_df.isna().sum())

print("\nDuplicate training headlines:")
print(train_df.duplicated(subset=["headline"]).sum())

##Handle Duplicates

In [ ]:
before_rows = len(train_df)

train_df = train_df.drop_duplicates(
    subset=["headline", "label"]
).reset_index(drop=True)

after_exact_pair_drop = len(train_df)

conflicting_duplicates = (
    train_df.groupby("headline")["label"]
    .nunique()
    .reset_index(name="label_count")
)

conflicting_duplicates = conflicting_duplicates[
    conflicting_duplicates["label_count"] > 1
]

if len(conflicting_duplicates) > 0:
    print("Warning: same headline appears with different labels.")
    print(conflicting_duplicates.head())

train_df = train_df.drop_duplicates(
    subset=["headline"],
    keep="first",
).reset_index(drop=True)

after_headline_drop = len(train_df)

print("Rows before duplicate handling:", before_rows)
print("Rows after duplicate label/headline pair removal:", after_exact_pair_drop)
print("Rows after headline-only duplicate removal:", after_headline_drop)

##Preprocess Text

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

train_cleaned_df = preprocess_dataframe(
    train_df,
    text_column="headline",
    lemmatizer=lemmatizer,
    stop_words=stop_words,
)

test_cleaned_df = preprocess_dataframe(
    test_df,
    text_column="headline",
    lemmatizer=lemmatizer,
    stop_words=stop_words,
)

In [ ]:
train_cleaned_df.head()

##Final Validation

In [ ]:
required_train_output_columns = {
    "label",
    "headline",
    "clean_text",
    "tokens_json",
    "processed_text",
}

required_test_output_columns = {
    "original_label",
    "headline",
    "clean_text",
    "tokens_json",
    "processed_text",
}

missing_train_output = required_train_output_columns - set(train_cleaned_df.columns)
missing_test_output = required_test_output_columns - set(test_cleaned_df.columns)

if missing_train_output:
    raise ValueError(f"Missing train output columns: {missing_train_output}")

if missing_test_output:
    raise ValueError(f"Missing test output columns: {missing_test_output}")

empty_processed_train = (train_cleaned_df["processed_text"].str.len() == 0).sum()
empty_processed_test = (test_cleaned_df["processed_text"].str.len() == 0).sum()

print("Empty processed train rows:", empty_processed_train)
print("Empty processed test rows:", empty_processed_test)

##Save Cleaned Data

In [ ]:
train_cleaned_df.to_csv(train_cleaned_path, index=False)
test_cleaned_df.to_csv(test_cleaned_path, index=False)

print(f"Saved cleaned train data to: {train_cleaned_path}")
print(f"Saved cleaned test data to: {test_cleaned_path}")

##Save Preprocessing Report

In [ ]:
report = {
    "raw_train_rows": int(before_rows),
    "clean_train_rows": int(len(train_cleaned_df)),
    "raw_test_rows": int(len(test_df)),
    "clean_test_rows": int(len(test_cleaned_df)),
    "duplicate_train_rows_removed": int(before_rows - len(train_cleaned_df)),
    "empty_processed_train_rows": int(empty_processed_train),
    "empty_processed_test_rows": int(empty_processed_test),
    "train_label_counts": {
        str(label): int(count)
        for label, count in train_cleaned_df["label"].value_counts().items()
    },
    "input_train_path": str(train_raw_path),
    "input_test_path": str(test_raw_path),
    "output_train_path": str(train_cleaned_path),
    "output_test_path": str(test_cleaned_path),
}

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)

print(f"Saved preprocessing report to: {report_path}")

##Reload Test

In [ ]:
reloaded_train = pd.read_csv(train_cleaned_path)
reloaded_test = pd.read_csv(test_cleaned_path)

print(reloaded_train.shape)
print(reloaded_test.shape)

reloaded_train.head()